### Basic working of NVIDIA NIM LLM in LangChain

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# NVIDIA_API_KEY is read from the environment (.env). Get a free key at https://build.nvidia.com
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0.1)

In [ ]:
poem = llm.invoke("Write a 4 line poem of my love for samosa")
print(poem.content)

In [ ]:
essay = llm.invoke("write email requesting refund for electronic item")
print(essay.content)

### Now let's load data from the FAQ csv file

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path='codebasics_faqs.csv', source_column="prompt")

# Store the loaded data in the 'data' variable
data = loader.load()

### NVIDIA NIM Embeddings

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

# Initialize NVIDIA NIM retrieval-optimized embeddings
embeddings = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5")

e = embeddings.embed_query("What is your refund policy?")

In [ ]:
len(e)

In [ ]:
e[:5]

The embedding for "What is your refund policy?" is a fixed-length vector (1024 dimensions for `nv-embedqa-e5-v5`). These numbers capture the semantic meaning of the text so similar questions land close together in vector space.

### Vector store using FAISS

In [ ]:
from langchain_community.vectorstores import FAISS

# Create a FAISS instance for vector database from 'data'
vectordb = FAISS.from_documents(documents=data,
                                embedding=embeddings)

# Create a retriever for querying the vector database
retriever = vectordb.as_retriever(score_threshold=0.7)

In [ ]:
rdocs = retriever.invoke("how about job placement support?")
rdocs

The retriever, built with FAISS and NVIDIA NIM embeddings, pulls the most relevant documents from our CSV knowledge store.

### Create RetrievalQA chain along with prompt template 🚀

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}

from langchain_classic.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm=llm,
                                    chain_type="stuff",
                                    retriever=retriever,
                                    input_key="query",
                                    return_source_documents=True,
                                    chain_type_kwargs=chain_type_kwargs)

### We are all set 👍🏼 Let's ask some questions now

In [ ]:
chain.invoke('Do you provide job assistance and also do you provide job gurantee?')

In [ ]:
chain.invoke("Do you guys provide internship and also do you offer EMI payments?")

In [ ]:
chain.invoke("do you have javascript course?")

In [ ]:
chain.invoke("should I learn power bi or tableau?")

In [ ]:
chain.invoke("I've a MAC computer. Can I use powerbi on it?")